# NB 60 — Subspace qualitative gallery (pooled vs hierarchical)

The subspace counterpart of the frame gallery **NB 47**: same validated-pool
quads, same prompts, same greedy top-k recipe (`k=4`, `STEPS=32`, λ=0), same
n=1→4 blend progression and the λ=0 vs λ=0.4 contrast — but generated with the
`subspace/` pipeline, once per concept construction (**pooled** and
**hierarchical**), so the decoded text of the two builds can be read side by side
and against NB 47's frame output.

Quads (all NB 32 pool members, identical to NB 47):

- **novel** — sorrow → fury → flood → passing (*"Write the opening of a novel."*)
- **traveler** — country → town → dwelling → respite (*"Tell me a short story." / "The traveler arrived at"*)

Concepts use the same policy as the quantitative Llama runs (8 languages,
`min_lemmas=11`). Blends use `ConceptSubspace.blend`. Output:
`resources/60_subspace_gallery.jsonl` + a formatted-markdown gallery print.

In [ ]:
import json
from pathlib import Path

import torch

from frames.evaluation.harness import EvaluationHarness
from frames.nlp import SupportedLanguages
from frames.utils.memory import gc_cuda
from subspace import ConceptSubspace, SubspaceUnembeddingRepresentation

In [ ]:
MAX_TOKENS = 3
K = 4
STEPS = 32  # gallery setting (matches NB 47; quantitative runs use 24)
MIN_LEMMAS = 11  # Llama policy, 8 languages
RANK, R_LEMMA = 8, 1
ARMS = ["pooled", "hierarchical"]
LAMBDAS = (0.0, 0.4)  # contrast pair, lead block only
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"

POOL = json.loads(Path("resources/32_concept_pool_bf16.json").read_text())
POOL_DELTA = {e["synset"]: round(e["delta"], 2) for e in POOL["pool"]}

QUADS = {
    "novel": ["sorrow.n.01", "fury.n.01", "flood.n.01", "passing.n.02"],
    "traveler": ["country.n.02", "town.n.02", "dwelling.n.01", "respite.n.04"],
}
assert all(m in POOL_DELTA for q in QUADS.values() for m in q)


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


PROMPTS = {
    "novel": chat("Write the opening of a novel.", ""),
    "traveler": chat("Tell me a short story.", "The traveler arrived at"),
}

LOG_PATH = Path("resources/60_subspace_gallery.jsonl")
LOG_PATH.unlink(missing_ok=True)
BASELINES_PATH = Path("resources/60_baselines.json")

In [ ]:
with gc_cuda():
    sur = SubspaceUnembeddingRepresentation.from_model_id(
        MODEL_ID,
        rank=RANK,
        r_lemma=R_LEMMA,
        language_codes=SupportedLanguages.Llama,  # 8-language FRH set
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

    # Baselines are arm-independent (model only) — one per block.
    baselines = {}
    for block, prompt in PROMPTS.items():
        baselines[block] = sur.generate([prompt], max_new_tokens=STEPS + 2, do_sample=False)[0]
    BASELINES_PATH.write_text(json.dumps(baselines, indent=2))

    for arm in ARMS:
        # Fresh harness per arm (expression cache keys on (text, synset) only).
        harness = EvaluationHarness(sur, LOG_PATH, MIN_LEMMAS, MAX_TOKENS)
        harness.member_lemmas(next(iter(POOL_DELTA)))  # force WordNet load out of loops

        concept = {
            name: sur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS, method=arm)
            for name in sorted({m for q in QUADS.values() for m in q})
        }

        for block, quad in QUADS.items():
            prompt = [PROMPTS[block]]
            base = [baselines[block]]

            def run(names, lam, kind):
                guide = (
                    concept[names[0]]
                    if len(names) == 1
                    else ConceptSubspace.blend([concept[m] for m in names])
                )
                torch.cuda.empty_cache()
                texts, probe = sur.generate_with_topk_guide(
                    prompt, guide=guide, k=K, steps=STEPS, fluency_weight=lam
                )
                harness.evaluate(
                    prompt,
                    texts,
                    names,
                    config={
                        "model": MODEL_KEY, "arm": arm, "block": block,
                        "n": len(names), "lam": lam, "kind": kind,
                    },
                    probe=probe,
                    baseline_texts=base,
                    concepts={m: concept[m] for m in names},
                    angles=False,
                )

            # n = 1..4 progression at λ=0 (canonical greedy)
            for n in range(1, 5):
                run(quad[:n], 0.0, "progression")
            # λ contrast on the n=2 lead (λ=0 already logged above)
            run(quad[:2], 0.4, "contrast")
        print(f"{arm}: done")

In [ ]:
records = [json.loads(x) for x in LOG_PATH.read_text().strip().split("\n")]
baselines = json.loads(BASELINES_PATH.read_text())


def short(name):
    return name.split(".")[0]


def fmt(r):
    c = r["config"]
    names = QUADS[c["block"]][: c["n"]]
    deltas = {n: r["expression"][n]["delta"] for n in names}
    ok = min(deltas.values()) > 0
    dstr = ", ".join(f"{short(n)} {d:+.2f}" for n, d in deltas.items())
    mark = "✓ all improved" if ok else "✗ partial"
    lam = f", λ={c['lam']}" if c["kind"] == "contrast" else ""
    return (
        f"**n={c['n']}: {' + '.join(short(n) for n in names)}{lam}** — "
        f"Δ [{dstr}] — {mark}\n> {r['continuation']}\n"
    )


lines = []
for block, quad in QUADS.items():
    arrow = " → ".join(short(m) for m in quad)
    pool_line = " · ".join(f"{short(m)} {POOL_DELTA[m]}" for m in quad)
    b = baselines[block]
    cont = b.split("assistant<|end_header_id|>")[-1] if "assistant" in b else b
    lines.append(f"## Block — {block}: {arrow}")
    lines.append(f"*All validated pool members. Singles Δ (nb32): {pool_line}.*\n")
    lines.append(f"> **Unsteered baseline** — {cont}\n")
    for arm in ARMS:
        lines.append(f"### {arm}")
        for r in records:
            if r["config"]["block"] == block and r["config"]["arm"] == arm:
                lines.append(fmt(r))
gallery = "\n".join(lines)
Path("resources/60_subspace_gallery.md").write_text(gallery)
print(gallery)